In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from conv import *
from converters import *
from matmul import *
from model import *
from pooling import *
from sgposit.pcposit import PCPosit
# Load MNIST test set
transform = transforms.Compose([transforms.ToTensor()])
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=False)

In [ ]:
# Load standard model
model = LeNetAvgPool()
model.load_state_dict(torch.load("model_avg.pth", map_location=torch.device('cpu')))
model.eval()

n, es = 8, 1  # Posit parameters
correct_standard = 0
correct_posit = 0
match_top1 = 0
top5_correct_posit = 0

In [ ]:
print("\n---- Inference Comparison ----\n")

for idx, (images, labels) in enumerate(testloader):
    if idx >= 10000:
        break

    # Standard FP32 Inference
    output_standard = model(images)
    pred_standard = torch.argmax(output_standard, dim=1).item()
    top5_standard = torch.topk(output_standard, 5).indices[0].tolist()

    # Convert image to posit input
    image_np = images.squeeze(0).numpy()  # shape [1, 28, 28]
    input_list = [[
        [PCPosit(int(float_to_posit(n, es, pixel), 2), mode='bits', nbits=n, es=es) for pixel in row]
        for row in image_np[0]
    ]]

    # Posit Inference
    output_posit = model_inference(input_list, n=n, es=es)
    output_posit_float = [val for val in output_posit]
    pred_posit = output_posit_float.index(max(output_posit_float))

    # Compare results
    correct_standard += (pred_standard == labels.item())
    correct_posit += (pred_posit == labels.item())
    match_top1 += (pred_posit == pred_standard)

    # Top-5 for posit
    top5_posit = sorted(range(len(output_posit_float)), key=lambda i: output_posit_float[i], reverse=True)[:5]
    top5_correct_posit += int(labels.item() in top5_posit)

    print(f"Sample #{idx + 1}")
    print(f"True Label          : {labels.item()}")
    print(f"Standard Prediction : {pred_standard}, Top-5: {top5_standard}")
    print(f"Posit Prediction    : {pred_posit}, Top-5: {top5_posit}")
    print(f"Match with FP32 Top-1: {'✅' if pred_posit == pred_standard else '❌'}")
    print(f"Top-5 Match with True Label  : {'✅' if labels.item() in top5_posit else '❌'}\n")

In [ ]:
# Final Stats
print("\n------ Summary ------")
print(f"Standard Accuracy on 10000 samples : {correct_standard}/10000")
print(f"Posit Accuracy on 10000 samples    : {correct_posit}/10000")
print(f"Top-5 Accuracy (Posit)          : {top5_correct_posit}/10000")
print(f"Top-1 Match with FP32           : {match_top1}/10000")